# ResNet-20 / CIFAR-10: input corruption, pooled by class

The CIFAR-10 analogue of `lenet_pixel_noise.ipynb`. Two independent
corruption sweeps over the seeded CIFAR-10 test subset, four models
compared:

- **MAP (pruned+refit)** -- the point estimate the samplers' `x_ref` /
  cold-start mask were derived from. Pipeline: `resnet20_pretrain.py` (pure
  SGD) -> `resnet20_reference.py` MAP refinement (Adam, Gaussian log-prior
  in the loss) -> prune 52.9% of coords -> refit the survivors. The weights
  used here are this checkpoint's `x_ref` vector (52.9% of entries exactly
  zero); its frozen BatchNorm running stats come from `module_state_dict`.
  ckpt full-test acc ~0.91.
- **SGD (frequentist)** -- the pure-SGD pretrained ResNet-20
  (`resnet20_pretrain.py`, plain SGD + momentum, cross-entropy only, no
  prior, dense). This is the checkpoint `resnet20_reference.py` *starts its
  MAP refinement from*, so it is the genuine frequentist baseline (Izmailov
  et al. 2021 Fig 15 role). ckpt full-test acc ~0.84.
- **Sticky Zig-Zag** and **Sticky Boomerang** posteriors.

What happens between SGD and the MAP: a Gaussian prior is added (the mode
shifts toward the prior mean), then half the weights are pruned and the
rest refit -- the refit is what lifts accuracy from ~0.84 to ~0.91. The two
checkpoints also carry different BatchNorm running stats, so MAP and SGD
are each pushed through their own module (`_module_for` in the engine cell);
running one's weights through the other's BN buffers collapses accuracy to
chance.

MAP and SGD flow through the pipeline as single-draw "posteriors" (dotted
lines); the samplers use `N_DRAWS_POOL` draws (solid).

1. **Synthetic Gaussian pixel noise** -- add i.i.d. N(0, sigma^2) per
   pixel-channel in de-normalised [0, 1] space at `SIGMAS` levels, then clip
   to [0, 1] (the CIFAR-10-C convention). Fine-grained, one corruption type.
2. **CIFAR-10-C** (Hendrycks & Dietterich 2019) -- the released corrupted
   test set: `CORRUPTIONS` types x 5 severities each. This is what Ovadia et
   al. / Izmailov et al. use. Requires `datasets/CIFAR-10-C/*.npy` on disk
   (see cell 1 for the download line); the CIFAR-10-C sweep self-skips if
   the files are missing.

Both sweeps: for each class in `CLASSES` pool `N_PER_CLASS` test images,
apply the corruption, push draws (or the single point vector) through,
aggregate P(class) over the pool plus accuracy / P(true) / confidence /
entropy / ECE (per class and pooled). Each bar is the posterior mean
(1/S) sum_s P(k | x, theta_s); whiskers are the 10-90% band across the
image pool.

**Subsetting.** CIFAR-10-C `.npy` files hold the *full* 10k CIFAR-10 test
set in torchvision's standard order. `load_cifar10_subset` draws its test
subset with `np.random.default_rng(42)`; we replay that exact draw to
recover the underlying 0..9999 test indices, then subsample `N_PER_CLASS`
per class from those -- so the CIFAR-10-C images line up 1:1 with the clean
ones. `N_TEST` in the config controls how large that subset is (1000 = the
size the ResNet sampling run used; 10000 = the whole test set, much slower).


## 1. Config

In [ ]:
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

if Path.cwd().name == "notebooks":
    os.chdir("..")
print("cwd:", Path.cwd())

DEVICE = (torch.device("mps") if torch.backends.mps.is_available()
          else torch.device("cuda") if torch.cuda.is_available()
          else torch.device("cpu"))
DTYPE = torch.float32
print("device:", DEVICE)

plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "font.size": 11, "figure.dpi": 120,
})

RUN_DIR = Path("results/paper/cifar")
RUN_SPECS = [("zigzag", "grid_sticky_zigzag.pt"),
             ("boomerang", "grid_sticky_boomerang.pt")]   # loaded if present
RUN_DISPLAY = {"zigzag": "Sticky Zig-Zag", "boomerang": "Sticky Boomerang",
               "map": r"$\beta_{\mathrm{ref}}$", "sgd": "SGD"}

# The pruned + refitted MAP -- the same file the samplers' x_ref / cold-start
# mask were derived from. Added to `runs` below as a 1-draw model so it is
# plotted, not just used as the consistency-check reference.
MAP_REF_PATH = Path("results/maps/resnet/resnet20_reference_N50000_steps2000_v2_pruned_refit.pt")
# Pure-SGD pretrained ResNet-20. resnet20_pretrain.py trains plain SGD with
# no prior; resnet20_reference.py then does MAP refinement starting from it,
# so this checkpoint IS the frequentist baseline (Izmailov et al. 2021 Fig
# 15 role) -- no separate SGD script needed. It carries its OWN BatchNorm
# running stats and is run through its own module (see cell 6 / cell 8).
SGD_REF_PATH = Path("results/maps/resnet/resnet20_pretrain_N50000_epochs80.pt")

CIFAR10_CLASSES = ["plane", "car", "bird", "cat", "deer",
                   "dog", "frog", "horse", "ship", "truck"]

CLASSES         = ["cat", "dog", "bird", "ship"]   # names or ints; which classes get pooled
N_PER_CLASS     = 100
N_DRAWS_POOL    = 1_000
POOL_SEED       = 0                    # which images get subsampled
NOISE_SEED      = 12345               # one seeded noise field per (class, sigma)

# --- sweep A: synthetic Gaussian noise ---
SIGMAS = [0.0, 0.01, 0.025, 0.05, 0.075, 0.1]   # on de-normalised [0,1]

# --- sweep B: CIFAR-10-C ---
CIFAR10C_DIR = Path("datasets/CIFAR-10-C")
# download:  mkdir -p datasets/CIFAR-10-C && cd datasets/CIFAR-10-C && \
#   curl -L -o CIFAR-10-C.tar "https://zenodo.org/record/2535967/files/CIFAR-10-C.tar?download=1" && \
#   tar xf CIFAR-10-C.tar --strip-components=1
CORRUPTIONS = ["gaussian_noise", "shot_noise", "motion_blur", "fog",
               "brightness", "contrast", "jpeg_compression", "elastic_transform"]
SEVERITIES  = [1, 2, 3, 4, 5]

CIFAR10_MEAN = torch.tensor((0.4914, 0.4822, 0.4465)).view(3, 1, 1)
CIFAR10_STD  = torch.tensor((0.2470, 0.2435, 0.2616)).view(3, 1, 1)
PRIOR_STD_W, PRIOR_STD_B, PRIOR_STD_BN_W, FAN_IN_SCALING = 2.0, 2.0, 1.0, True
BASE_SEED = 42
N_TEST = 1_000

SAVE_DIR = Path("results/paper/CIFAR")
# SAVE_DIR.mkdir(parents=True, exist_ok=True)   # uncomment when saving

TRUE_GREEN = "#2CA02C"
SAMPLER_COLORS = {"zigzag": "#4C72B0", "boomerang": "#DD8452",
                  "map": "0.35", "sgd": "0.55"}

CLASS_IDS = [c if isinstance(c, int) else CIFAR10_CLASSES.index(c) for c in CLASSES]
CLASS_NAME = {i: CIFAR10_CLASSES[i] for i in CLASS_IDS}
print("classes:", [(i, CLASS_NAME[i]) for i in CLASS_IDS])


## 2. Load the models, the data, and the predictive engine

`runs` ends up with up to four entries in a fixed order: `map`, `sgd`
(1-draw point estimates, each with its own `module_state_dict`), then
`zigzag`, `boomerang`. `X_test` / `y_test` are the seeded 1k subset (same as
the run). `TEST_IDX` is the underlying 0..9999 index of each of those 1000
images in torchvision's test order -- the key to lining CIFAR-10-C up with
the clean images.


In [ ]:
runs = {}
for label, fname in RUN_SPECS:
    p = RUN_DIR / fname
    if not p.exists():
        print(f"[{label}] missing {p} -- skipped"); continue
    runs[label] = torch.load(p, map_location="cpu", weights_only=False)
    ck = runs[label]
    print(f"[{label}] {ck['samples'].shape[0]} draws  test_acc(ckpt)={ck['test_accuracy']:.4f}  "
          f"sparsity(ckpt)={ck['sparsity_frac']:.4f}")
assert runs, f"no run files under {RUN_DIR}"

map_ck = torch.load(MAP_REF_PATH, map_location="cpu", weights_only=False)
MSD = map_ck["module_state_dict"]                 # frozen BN running stats
X_REF = map_ck["x_ref"].to(DTYPE)
D = int(X_REF.shape[0])
print(f"\nMAP ref: {MAP_REF_PATH.name}  D={D}  sparsity={float((X_REF == 0).float().mean()):.4f}")
for label, ck in runs.items():
    dmax = (ck["x_ref"].to(DTYPE) - X_REF).abs().max().item()
    cs_ok = torch.equal(ck["cold_start_mask"].bool(), (X_REF == 0))
    assert dmax < 1e-5 and cs_ok, (
        f"[{label}] MAP MISMATCH (max|delta x_ref|={dmax:.2e}, cold_start match={cs_ok})."
    )
    print(f"  [{label}] matches run x_ref (max|delta|={dmax:.1e}) and cold_start_mask OK")

# --- point-estimate models -----------------------------------------------
# Each flows through the pipeline as a 1-draw "posterior". Both carry their
# OWN module_state_dict (BatchNorm running stats differ from each other and
# from the samplers' MSD), consumed by _predict_and_aggregate in cell 8.
_order = ["map", "sgd", "zigzag", "boomerang"]

runs["map"] = {"samples": X_REF.unsqueeze(0), "module_state_dict": MSD}
print(f"\nMAP  -> runs['map']  (1 draw, pruned+refit, ckpt test_acc={map_ck['test_acc']:.4f})")

if SGD_REF_PATH.exists():
    sgd_ck = torch.load(SGD_REF_PATH, map_location="cpu", weights_only=False)
    assert sgd_ck.get("architecture") == "resnet20", "SGD checkpoint architecture mismatch"
    from sazz.gpu_friendly.models.neural_networks import ResNet20 as _ResNet20_probe
    _m = _ResNet20_probe(activation="relu")
    _m.load_state_dict(sgd_ck["state_dict"])
    W_SGD = torch.cat([p.detach().flatten() for p in _m.parameters()]).to(DTYPE)
    assert W_SGD.shape[0] == D, f"SGD flat D={W_SGD.shape[0]} != {D}"
    runs["sgd"] = {"samples": W_SGD.unsqueeze(0), "module_state_dict": sgd_ck["state_dict"]}
    print(f"SGD  -> runs['sgd']  (1 draw, pure SGD, ckpt test_acc={sgd_ck['test_acc']:.4f}, "
          f"epochs={sgd_ck['n_epochs']})")
    del _m
else:
    print(f"[sgd] missing {SGD_REF_PATH} -- skipped")

runs = {k: runs[k] for k in _order if k in runs}
print("models:", list(runs))


In [ ]:
from sazz.gpu_friendly.scripts.fast_cifar_resnet import load_cifar10_subset

_data = load_cifar10_subset(50_000, N_TEST, BASE_SEED, Path("datasets"),
                            dtype=DTYPE, device="cpu")
X_test, y_test = _data["X_test"], _data["y_test"]   # [N,3,32,32] normalised, CPU

# load_cifar10_subset draws train_idx FIRST, then test_idx from the SAME rng,
# so replaying just the test draw is wrong -- the train draw offsets the
# stream. Replay both to recover the underlying 0..9999 torchvision test
# indices of these 1000 images.
_rng = np.random.default_rng(BASE_SEED)
_ = _rng.choice(50_000, size=50_000, replace=False)          # consume the train draw
TEST_IDX = _rng.choice(10_000, size=N_TEST, replace=False)   # the test draw we want

# sanity: those raw test images, normalised, must equal X_test
from torchvision import datasets as _tvd, transforms as _tvt
_raw_test = _tvd.CIFAR10("datasets", train=False, download=True,
                         transform=_tvt.Compose([_tvt.ToTensor(),
                             _tvt.Normalize(tuple(CIFAR10_MEAN.flatten().tolist()),
                                            tuple(CIFAR10_STD.flatten().tolist()))]))
_chk = torch.stack([_raw_test[int(k)][0] for k in TEST_IDX[:32]])
assert torch.allclose(_chk, X_test[:32], atol=1e-5), "TEST_IDX replay does not match X_test"
_chk_y = torch.tensor([_raw_test[int(k)][1] for k in TEST_IDX[:32]])
assert torch.equal(_chk_y, y_test[:32].cpu()), "TEST_IDX label replay mismatch"
print("X_test:", tuple(X_test.shape), " TEST_IDX replay verified (images + labels)")
print("class counts in subset:", {CLASS_NAME[i]: int((y_test == i).sum()) for i in CLASS_IDS})

In [ ]:
from sazz.gpu_friendly.models.neural_networks import ResNet20
from sazz.gpu_friendly.models.model import BayesianModule
from sazz.gpu_friendly.models.priors import build_fan_in_prior_precision_resnet

_module = ResNet20(activation="relu").to(dtype=DTYPE, device=DEVICE)
_module.load_state_dict(MSD, strict=False)         # restore frozen BN running stats
_module.eval()
_prec = build_fan_in_prior_precision_resnet(_module, PRIOR_STD_W, PRIOR_STD_B,
                                            PRIOR_STD_BN_W, FAN_IN_SCALING,
                                            dtype=DTYPE, device=DEVICE)
_bm = BayesianModule.build(_module, likelihood="categorical",
                           X=X_test[:2].to(DEVICE), y=y_test[:2].to(DEVICE),
                           prior_precision=_prec, dtype=DTYPE, device=DEVICE)
_pdf = _bm.param_dict_fn


# Per-model module cache. The samplers share `_bm` (frozen BN stats = MSD).
# The MAP and SGD point estimates each carry their own module_state_dict --
# their BatchNorm running_mean/running_var differ, and running SGD weights
# through the MAP's BN buffers (or vice versa) mis-normalises every
# activation. _module_for builds one eval-mode ResNet20 per such state_dict
# once and reuses it.
_MODULE_CACHE = {}


def _module_for(state_dict):
    if state_dict is None:
        return _bm.module
    key = id(state_dict)
    if key not in _MODULE_CACHE:
        m = ResNet20(activation="relu").to(dtype=DTYPE, device=DEVICE)
        m.load_state_dict(state_dict, strict=False)
        m.eval()
        _MODULE_CACHE[key] = m
    return _MODULE_CACHE[key]


@torch.no_grad()
def predict_probs(beta, X, bs=256, module=None):
    module = module if module is not None else _bm.module
    beta = beta.to(dtype=DTYPE, device=DEVICE)
    out = []
    for i in range(0, X.shape[0], bs):
        xb = X[i:i + bs].to(dtype=DTYPE, device=DEVICE)
        out.append(torch.softmax(
            torch.func.functional_call(module, _pdf(beta), (xb,)), -1).cpu())
    return torch.cat(out)


def denorm(x_img_norm):
    return (x_img_norm * CIFAR10_STD + CIFAR10_MEAN).clamp(0, 1).permute(1, 2, 0).numpy()


def normalise_uint8(arr_hwc):
    # [...,32,32,3] uint8 -> [...,3,32,32] normalised float tensor
    t = torch.from_numpy(np.ascontiguousarray(arr_hwc).copy()).float().div(255.0)
    t = t.permute(*range(t.ndim - 3), t.ndim - 1, t.ndim - 3, t.ndim - 2)  # HWC -> CHW
    return (t - CIFAR10_MEAN) / CIFAR10_STD


## 3. Shared machinery

`run_noise_sweep()` sweeps `SIGMAS` (synthetic). `run_c_sweep(corruption)`
sweeps the 5 CIFAR-10-C severities of one corruption. Both return
`(levels, spans, big_X, agg)`. `agg[label]` has per-class entries
`(class, level)` (used by `bar_grid`) and a `("pool", level)` entry that
pools every class in `CLASSES` -- the paper figure `trajectories` uses only
the pooled entry (one curve per model, no per-animal split), matching
`lenet_pixel_noise.ipynb` but without the per-class lines.


In [ ]:
def _pool_indices():
    rng = np.random.default_rng(POOL_SEED)
    out = {}
    for i in CLASS_IDS:
        cand = np.where(y_test.numpy() == i)[0]      # positions within the 1k subset
        out[i] = rng.choice(cand, size=min(N_PER_CLASS, len(cand)), replace=False)
    return out


def _cal_metrics(block, i):
    """Per-image-pool metrics for one (model, class, level) cell.
    `block` is [n_img, 10] posterior-mean probs; `i` is the true class.
      conf     -- mean max-prob (confidence in the PREDICTED class).
                  conf > acc is overconfidence.
      entropy  -- mean predictive entropy (nats).
      ece      -- expected calibration error (15 equal-width conf bins).
    """
    pred = block.argmax(1); conf = block.max(1)
    correct = (pred == i).astype(float)
    ent = -(block * np.log(np.clip(block, 1e-12, 1.0))).sum(1)
    bins = np.linspace(0.0, 1.0, 16)
    bi = np.clip(np.digitize(conf, bins) - 1, 0, 14)
    n = len(block)
    ece = sum(((bi == b).sum() / n) * abs(correct[bi == b].mean() - conf[bi == b].mean())
              for b in range(15) if (bi == b).any())
    return dict(conf=float(conf.mean()), entropy=float(ent.mean()), ece=float(ece))


def _predict_and_aggregate(big_X, spans, levels):
    pooled = {}
    for label, ck in runs.items():
        S = ck["samples"]
        mod = _module_for(ck.get("module_state_dict"))
        g = torch.Generator().manual_seed(0)
        draw_idx = torch.randperm(S.shape[0], generator=g)[:min(N_DRAWS_POOL, S.shape[0])]
        acc = torch.zeros(big_X.shape[0], 10)
        for j in draw_idx:
            acc += predict_probs(S[j], big_X, module=mod)
        pooled[label] = acc / len(draw_idx)
    agg = {label: {} for label in runs}
    for label in runs:
        P = pooled[label]
        for i in CLASS_IDS:
            for lv in levels:
                a, b = spans[(i, lv)]
                block = P[a:b].numpy()
                agg[label][(i, lv)] = dict(
                    mean=block.mean(0),
                    lo=np.percentile(block, 10, axis=0),
                    hi=np.percentile(block, 90, axis=0),
                    acc=float((block.argmax(1) == i).mean()),
                    p_true=float(block[:, i].mean()),
                    n_img=block.shape[0],
                    **_cal_metrics(block, i),
                )
        # pooled over CLASSES: one curve per model per level (used by the
        # paper figure -- for CIFAR we do NOT split per class).
        for lv in levels:
            blocks, trues = [], []
            for i in CLASS_IDS:
                a, b = spans[(i, lv)]
                blocks.append(P[a:b].numpy()); trues.append(np.full(b - a, i))
            Pool = np.concatenate(blocks); tru = np.concatenate(trues)
            pred = Pool.argmax(1); cf = Pool.max(1); corr = (pred == tru).astype(float)
            ent = -(Pool * np.log(np.clip(Pool, 1e-12, 1.0))).sum(1)
            bins = np.linspace(0.0, 1.0, 16)
            bi = np.clip(np.digitize(cf, bins) - 1, 0, 14)
            ece = sum(((bi == b).sum() / len(Pool)) *
                      abs(corr[bi == b].mean() - cf[bi == b].mean())
                      for b in range(15) if (bi == b).any())
            agg[label][("pool", lv)] = dict(
                acc=float(corr.mean()), conf=float(cf.mean()),
                p_true=float(Pool[np.arange(len(Pool)), tru].mean()),
                entropy=float(ent.mean()), ece=float(ece), n_img=len(Pool),
            )
    return agg


def run_noise_sweep():
    pool_idx = _pool_indices()

    def add_noise(X_norm, sigma, seed):
        if sigma == 0.0:
            return X_norm.clone()
        g = torch.Generator().manual_seed(seed)
        img01 = X_norm * CIFAR10_STD + CIFAR10_MEAN
        img01 = (img01 + sigma * torch.randn(X_norm.shape, generator=g)).clamp(0, 1)
        return (img01 - CIFAR10_MEAN) / CIFAR10_STD

    big_X, spans, row = [], {}, 0
    for i in CLASS_IDS:
        Xc = X_test[pool_idx[i]]
        for si, sig in enumerate(SIGMAS):
            big_X.append(add_noise(Xc, sig, NOISE_SEED + 1000 * si + i))
            spans[(i, sig)] = (row, row + len(Xc)); row += len(Xc)
    big_X = torch.cat(big_X)
    agg = _predict_and_aggregate(big_X, spans, SIGMAS)
    print(f"[noise] done: {len(runs)} models x {len(CLASS_IDS)} classes x {len(SIGMAS)} sigmas")
    return SIGMAS, spans, big_X, agg


def run_c_sweep(corruption):
    npy = CIFAR10C_DIR / f"{corruption}.npy"
    if not npy.exists():
        raise FileNotFoundError(f"{npy} not found -- download CIFAR-10-C (see cell 1).")
    arr = np.load(npy, mmap_mode="r")                 # [50000,32,32,3] uint8; sev s = rows (s-1)*10000 : s*10000
    pool_idx = _pool_indices()
    # map each pooled subset-position back to its 0..9999 test index
    subset_to_test = {i: TEST_IDX[pool_idx[i]] for i in CLASS_IDS}

    levels = [0] + SEVERITIES                          # 0 = clean reference
    big_X, spans, row = [], {}, 0
    for i in CLASS_IDS:
        Xc_clean = X_test[pool_idx[i]]                 # already normalised
        for lv in levels:
            if lv == 0:
                Xc = Xc_clean.clone()
            else:
                rows = (lv - 1) * 10_000 + subset_to_test[i]
                Xc = normalise_uint8(np.asarray(arr[rows]).copy())
            big_X.append(Xc)
            spans[(i, lv)] = (row, row + len(Xc)); row += len(Xc)
    big_X = torch.cat(big_X)
    agg = _predict_and_aggregate(big_X, spans, levels)
    print(f"[C:{corruption}] done: {len(runs)} models x {len(CLASS_IDS)} classes x {len(levels)} levels")
    return levels, spans, big_X, agg


# ---- plotting (shared) ---------------------------------------------------
DIGITS = np.arange(10)
_POINT = ("map", "sgd")                       # 1-draw models: no posterior spread


def show_examples(title, levels, spans, big_X, level_fmt=lambda lv: f"{lv:g}"):
    fig, axes = plt.subplots(len(CLASS_IDS), len(levels),
                             figsize=(2.1 * len(levels), 2.1 * len(CLASS_IDS)), squeeze=False)
    for r, i in enumerate(CLASS_IDS):
        for cc, lv in enumerate(levels):
            a, _ = spans[(i, lv)]
            ax = axes[r][cc]
            ax.imshow(denorm(big_X[a]))
            ax.set_xticks([]); ax.set_yticks([])
            if r == 0:
                ax.set_title(level_fmt(lv) + ("  (clean)" if cc == 0 else ""), fontsize=10)
            if cc == 0:
                ax.set_ylabel(CLASS_NAME[i], fontsize=11)
    fig.suptitle(title, y=1.01); fig.tight_layout(); plt.show()


def bar_grid(title, xlabel, levels, agg, level_fmt=lambda lv: f"{lv:g}"):
    for label in runs:
        if label in _POINT:                   # point estimates have no spread to show
            continue
        fig, axes = plt.subplots(len(CLASS_IDS), len(levels),
                                 figsize=(3.1 * len(levels), 2.6 * len(CLASS_IDS)),
                                 squeeze=False, sharey=True)
        for r, i in enumerate(CLASS_IDS):
            for cc, lv in enumerate(levels):
                ax = axes[r][cc]
                A = agg[label][(i, lv)]
                ax.axvspan(i - 0.5, i + 0.5, color=TRUE_GREEN, alpha=0.15, zorder=0)
                edge = ["none"] * 10; lw = [0.0] * 10
                edge[i] = TRUE_GREEN; lw[i] = 2.0
                ax.bar(DIGITS, A["mean"], color=SAMPLER_COLORS.get(label, "#666"),
                       edgecolor=edge, linewidth=lw, width=0.78, zorder=2)
                ax.errorbar(DIGITS, A["mean"],
                            yerr=[np.clip(A["mean"] - A["lo"], 0, None),
                                  np.clip(A["hi"] - A["mean"], 0, None)],
                            fmt="none", ecolor="black", elinewidth=0.8, capsize=1.5, zorder=3)
                ax.axvline(i, color=TRUE_GREEN, lw=1.5, zorder=1)
                ax.set_xticks(DIGITS)
                ax.set_xticklabels(CIFAR10_CLASSES, rotation=60, ha="right", fontsize=6)
                ax.set_ylim(0, 1)
                for t in ax.get_xticklabels():
                    if t.get_text() == CLASS_NAME[i]:
                        t.set_color(TRUE_GREEN); t.set_fontweight("bold")
                ax.set_title(f"{CLASS_NAME[i]}, {xlabel}={level_fmt(lv)}  |  acc={A['acc']:.2f}, "
                             f"P(true)={A['p_true']:.2f}  (n={A['n_img']})", fontsize=8)
                if cc == 0:
                    ax.set_ylabel(CLASS_NAME[i], fontsize=11)
        fig.suptitle(f"{RUN_DISPLAY[label]}: {title}  "
                     f"(bars = mean over {N_DRAWS_POOL} draws + image pool, "
                     f"whiskers = 10-90% across images)", y=1.005, fontsize=12)
        fig.tight_layout(); plt.show()


def trajectories(xlabel, levels, agg, x_for_plot=None, title=None, save_as=None):
    """Paper figure, same layout as lenet_pixel_noise.ipynb's shift_figure
    but pooled over classes (no per-animal split). Rows = models
    (beta_ref / SGD as dotted point estimates, Zig-Zag / Boomerang solid).
    Columns, all from the ("pool", level) aggregates:
      1. accuracy (+ chance line)
      2. P(true class)
      3. accuracy vs mean top-class confidence (shaded gap = over/under-conf)
      4. predictive entropy (nats) (+ uniform-10 reference line)
    """
    xs = x_for_plot if x_for_plot is not None else levels
    labels = list(runs)
    col_titles = ["Accuracy", "P(true class)",
                  "Accuracy vs confidence", "Predictive entropy"]
    n_cls = 10

    nrows, ncols = len(labels), 4
    fig, axes = plt.subplots(nrows, ncols, figsize=(9.2, 1.85 * nrows + 0.7),
                             sharex="col", squeeze=False)
    fig.subplots_adjust(wspace=0.28, hspace=0.30,
                        left=0.08, right=0.99, top=0.86, bottom=0.14)

    for r, label in enumerate(labels):
        is_point = label in _POINT
        ls = ":" if is_point else "-"
        lw = 1.1 if is_point else 1.6
        col = SAMPLER_COLORS.get(label, "#666")

        acc = [agg[label][("pool", lv)]["acc"] for lv in levels]
        pt = [agg[label][("pool", lv)]["p_true"] for lv in levels]
        conf = [agg[label][("pool", lv)]["conf"] for lv in levels]
        ent = [agg[label][("pool", lv)]["entropy"] for lv in levels]

        ax = axes[r][0]
        ax.plot(xs, acc, marker="o", ms=3, lw=lw, ls=ls, color=col)
        ax.axhline(1 / n_cls, color="0.65", lw=0.7, ls="--")
        ax.set_ylim(-0.02, 1.02)

        ax = axes[r][1]
        ax.plot(xs, pt, marker="o", ms=3, lw=lw, ls=ls, color=col)
        ax.axhline(1 / n_cls, color="0.65", lw=0.7, ls="--")
        ax.set_ylim(-0.02, 1.02)

        ax = axes[r][2]
        ax.plot(xs, acc, marker="o", ms=3, lw=1.6, color="0.20", label="accuracy")
        ax.plot(xs, conf, marker="s", ms=3, lw=1.6, color="#C44E52", label="confidence")
        ax.fill_between(xs, acc, conf, color="#C44E52", alpha=0.12, lw=0)
        ax.set_ylim(-0.02, 1.02)
        if r == 0:
            ax.legend(fontsize=6.5, loc="lower left", frameon=False, handlelength=1.2)

        ax = axes[r][3]
        ax.plot(xs, ent, marker="o", ms=3, lw=1.6, color="#4C72B0")
        ax.axhline(np.log(n_cls), color="0.65", lw=0.7, ls="--")   # uniform-10 entropy
        ax.set_ylim(-0.05, np.log(n_cls) * 1.08)

        for j in range(ncols):
            axes[r][j].grid(alpha=0.25)
            axes[r][j].tick_params(labelsize=6.5, length=2, pad=1)
            if r == 0:
                axes[r][j].set_title(col_titles[j], fontsize=10, pad=4)
            if r == nrows - 1:
                axes[r][j].set_xlabel(xlabel, fontsize=10)
            else:
                axes[r][j].tick_params(labelbottom=False)
        axes[r][0].set_ylabel(RUN_DISPLAY.get(label, label), fontsize=10)

    if title:
        fig.suptitle(title, fontsize=10, y=0.96)
    if save_as is not None:
        fig.savefig(SAVE_DIR / f"{save_as}.png", dpi=300, bbox_inches="tight")
        fig.savefig(SAVE_DIR / f"{save_as}.pdf", bbox_inches="tight")
    plt.show()


## 4. Sweep A -- synthetic Gaussian pixel noise

In [ ]:
# noi_levels, noi_spans, noi_X, noi_agg = run_noise_sweep()
# show_examples("synthetic Gaussian noise: one example per class per sigma",
#               noi_levels, noi_spans, noi_X, level_fmt=lambda s: f"sigma={s:g}")

In [ ]:
# bar_grid("pooled P(class) vs synthetic Gaussian noise", "sigma", noi_levels, noi_agg,
#          level_fmt=lambda s: f"{s:g}")

In [ ]:
# trajectories("pixel-noise sigma", noi_levels, noi_agg, save_as="cifar_noise_paper")

## 5. Sweep B -- CIFAR-10-C

One corruption at a time (`C_PICK`). Levels are `0` (clean) then severities
1-5. Re-run the three cells with a different `C_PICK` for each corruption in
`CORRUPTIONS`; or loop over all of them in the last cell.

In [ ]:
C_PICK = "gaussian_noise"      # any name from CORRUPTIONS

try:
    c_levels, c_spans, c_X, c_agg = run_c_sweep(C_PICK)
    _HAVE_C = True
except FileNotFoundError as e:
    print(e); print("-> skipping CIFAR-10-C sweeps. Download it (cell 1) and re-run.")
    _HAVE_C = False
    
if _HAVE_C:
    show_examples(f"CIFAR-10-C '{C_PICK}': one example per class per severity",
                  c_levels, c_spans, c_X, level_fmt=lambda s: "clean" if s == 0 else f"sev {s}")

In [ ]:
C_PICK = "gaussian_noise"
if _HAVE_C:
    bar_grid(f"pooled P(class) vs CIFAR-10-C '{C_PICK}' severity", "sev",
             c_levels, c_agg, level_fmt=lambda s: "0" if s == 0 else str(s))

In [ ]:
C_PICK = "gaussian_noise"
if _HAVE_C:
    trajectories(f"severity", c_levels, c_agg, title="Gaussian noise corruption",
                 save_as=f"cifar_C_{C_PICK}_paper")

In [ ]:
C_PICK = "fog"      # any name from CORRUPTIONS

try:
    c_levels, c_spans, c_X, c_agg = run_c_sweep(C_PICK)
    _HAVE_C = True
except FileNotFoundError as e:
    print(e); print("-> skipping CIFAR-10-C sweeps. Download it (cell 1) and re-run.")
    _HAVE_C = False
    
if _HAVE_C:
    show_examples(f"CIFAR-10-C '{C_PICK}': one example per class per severity",
                  c_levels, c_spans, c_X, level_fmt=lambda s: "clean" if s == 0 else f"sev {s}")

In [ ]:
C_PICK = "fog"
if _HAVE_C:
    bar_grid(f"pooled P(class) vs CIFAR-10-C '{C_PICK}' severity", "sev",
             c_levels, c_agg, level_fmt=lambda s: "0" if s == 0 else str(s))

In [ ]:
C_PICK = "fog"
if _HAVE_C:
    trajectories(f"severity", c_levels, c_agg, title="Fog corruption",
                 save_as=f"cifar_C_{C_PICK}_paper")

In [ ]:
C_PICK = "brightness"      # any name from CORRUPTIONS

try:
    c_levels, c_spans, c_X, c_agg = run_c_sweep(C_PICK)
    _HAVE_C = True
except FileNotFoundError as e:
    print(e); print("-> skipping CIFAR-10-C sweeps. Download it (cell 1) and re-run.")
    _HAVE_C = False
    
if _HAVE_C:
    show_examples(f"CIFAR-10-C '{C_PICK}': one example per class per severity",
                  c_levels, c_spans, c_X, level_fmt=lambda s: "clean" if s == 0 else f"sev {s}")

In [ ]:
C_PICK = "brightness"   
if _HAVE_C:
    bar_grid(f"pooled P(class) vs CIFAR-10-C '{C_PICK}' severity", "sev",
             c_levels, c_agg, level_fmt=lambda s: "0" if s == 0 else str(s))

In [ ]:
C_PICK = "brightness"   
if _HAVE_C:
    trajectories(f"severity", c_levels, c_agg, title="Brightness corruption",
                 save_as=f"cifar_C_{C_PICK}_paper")